In [29]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque

# --- ENVIRONMENT ---
VOCABULARY = ['I', 'love', 'you', 'too', 'are', 'really', 'incredible']
GOALS = ['I love you', 'you are really incredible']

# VOCABULARY = ['I', 'love', 'you', 'me']
# GOALS = ['I love you', 'you love me']

class TokenEnv:
    def __init__(self, actions, goal_states):
        self._actions = actions
        self._goal_states = goal_states
        # Map tokens to indices for the neural network
        self.tokenizer = {token: i + 1 for i, token in enumerate(actions)} # 0 is padding
        self.tokenizer[''] = 0
        self.vocab_size = len(actions) + 1

    def get_actions(self) -> list:
        return self._actions

    @staticmethod
    def step(state: str, action: str) -> str:
        if state == "":
            return action
        return f'{state} {action}'

    def is_goal(self, state: str) -> bool:
        return state in self._goal_states

    def is_terminal(self, state: str) -> bool:
        return self.is_goal(state) or len(state.split()) >= 5

    def expand(self, state: str) -> list:
        """Returns a list of tuples (next_state, step_cost) for all possible actions."""
        results = []
        for action in self.get_actions():
            next_state = self.step(state, action)
            results.append((next_state, 1.0)) # Step cost is 1
        return results

    def state_to_tensor(self, state: str, max_len: int = 5) -> torch.Tensor:
        tokens = state.split() if state else []
        # Truncate to max_len so input dimension is ALWAYS consistent
        ids = [self.tokenizer.get(t, 0) for t in tokens][:max_len]
        padded_ids = ids + [0] * (max_len - len(ids))
        return torch.tensor(padded_ids).unsqueeze(0)


In [30]:
# --- MODEL ARCHITECTURE ---
class TokenHeurModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=16, hidden_dim=32):
        super(TokenHeurModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.fc1 = nn.Linear(embedding_dim * 5, hidden_dim) # 5 tokens max
        self.fc2 = nn.Linear(hidden_dim, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        # x shape: [Batch, Sequence_Len]
        embedded = self.embedding(x) # [Batch, Seq, Emb]
        flattened = embedded.view(embedded.size(0), -1) # Flatten tokens
        x = self.relu(self.fc1(flattened))
        x = self.fc2(x)
        # CRITICAL: Distance cannot be negative! Clamps output to >= 0
        return torch.relu(x) 


In [31]:
class HeuristicTrainer:
    def __init__(self, env, model, lr=1e-3):
        self.env = env
        self.model = model
        
        # 1. Establish the Target Network (Deepcopy of main model)
        self.target_model = copy.deepcopy(model)
        self.target_model.eval()
        
        self.optimizer = optim.Adam(model.parameters(), lr=lr)
        self.criterion = nn.MSELoss()
        
        # 2. Increase buffer size so rare goals aren't forgotten!
        self.replay_buffer = deque(maxlen=100000) 
        self.train_steps = 0
    def add_to_buffer(self, state, is_solved):
        neighbors = self.env.expand(state)
        self.replay_buffer.append((state, is_solved, neighbors))
    def update_target_network(self):
        # Sync the weights
        self.target_model.load_state_dict(self.model.state_dict())
    def train_step(self, batch_size=32):
        if len(self.replay_buffer) < batch_size:
            return None
        batch = random.sample(self.replay_buffer, batch_size)
        states, is_solved_list, neighbors_batch = zip(*batch)
        inputs = torch.cat([self.env.state_to_tensor(s) for s in states])
        targets = []
        with torch.no_grad():
            for i, state in enumerate(states):
                if is_solved_list[i]:
                    targets.append(0.0) 
                elif self.env.is_terminal(state):
                    targets.append(10.0) # Dead end penalty
                else:
                    n_states = [n[0] for n in neighbors_batch[i]]
                    n_costs = [n[1] for n in neighbors_batch[i]]
                    
                    n_inputs = torch.cat([self.env.state_to_tensor(ns) for ns in n_states])
                    
                    # 3. USE TARGET NETWORK FOR PREDICTIONS
                    n_values = self.target_model(n_inputs).squeeze().tolist()
                    
                    target_val = min([c + nv for c, nv in zip(n_costs, n_values)])
                    targets.append(target_val)
        targets_tensor = torch.tensor(targets).unsqueeze(1).float()
        self.model.train()
        self.optimizer.zero_grad()
        predictions = self.model(inputs)
        loss = self.criterion(predictions, targets_tensor)
        loss.backward()
        self.optimizer.step()
        
        self.train_steps += 1
        return loss.item()

In [36]:
# --- MAIN LOOP ---
def run_training():
    env = TokenEnv(VOCABULARY, GOALS)
    model = TokenHeurModel(env.vocab_size)
    trainer = HeuristicTrainer(env, model)

    print("Starting Token Heuristic Training (Value-Based)...")

    total_episodes = 50001
    for ep in range(total_episodes):
        # 1. Collect Data (Exploration via random walk)
        state = ""

        # Epsilon decay from 1.0 down to 0.1 over half the training
        epsilon = max(0.1, 1.0 - (ep / (total_episodes * 0.5)))
        
        while True:
            # Add the state BEFORE taking an action (or immediately after breaking)
            trainer.add_to_buffer(state, env.is_goal(state))
            
            if env.is_terminal(state):
                break
            
            # --- Epsilon-Greedy Action Selection ---
            if random.random() < epsilon:
                action = random.choice(VOCABULARY)
            else:
                neighbors = env.expand(state)
                n_states = [n[0] for n in neighbors]
                n_inputs = torch.cat([env.state_to_tensor(ns) for ns in n_states])
                
                with torch.no_grad():
                    n_values = trainer.target_model(n_inputs).squeeze(-1).tolist()
                
                best_idx = n_values.index(min(n_values))
                action = env.get_actions()[best_idx]
            
            state = env.step(state, action)

        # Sync target network every 500 steps (arbitrary choice)
        if trainer.train_steps % 500 == 0:
            trainer.update_target_network()

        # 2. Train
        loss = trainer.train_step(batch_size=32)

        # 3. Logging every 50 episodes
        if ep % 50 == 0:
            avg_h = 0
            with torch.no_grad():
                test_states = ["", "I", "I love", "you are", "you are really"]
                test_tensors = torch.cat([env.state_to_tensor(ts) for ts in test_states])
                preds = model(test_tensors).squeeze().tolist()
                avg_h = sum(preds) / len(preds)

            print(f"Episode {ep:03d} | Loss: {loss if loss else 0.0:.4f} | Avg Predicted H: {avg_h:.2f}")

    # Final Validation
    print("\nFinal Model Predictions:")
    for goal in GOALS:
        h = model(env.state_to_tensor(goal)).item()
        print(f" Goal: '{goal}' -> Predicted H: {h:.4f}")

    return model


In [37]:
model = run_training()

Starting Token Heuristic Training (Value-Based)...
Episode 000 | Loss: 0.0000 | Avg Predicted H: 0.06
Episode 050 | Loss: 9.1792 | Avg Predicted H: 1.20
Episode 100 | Loss: 3.3628 | Avg Predicted H: 1.52
Episode 150 | Loss: 3.5877 | Avg Predicted H: 1.14
Episode 200 | Loss: 2.7151 | Avg Predicted H: 0.95
Episode 250 | Loss: 1.2086 | Avg Predicted H: 0.64
Episode 300 | Loss: 0.2972 | Avg Predicted H: 0.74
Episode 350 | Loss: 0.3204 | Avg Predicted H: 0.81
Episode 400 | Loss: 0.1168 | Avg Predicted H: 0.84
Episode 450 | Loss: 0.0630 | Avg Predicted H: 0.88
Episode 500 | Loss: 0.0319 | Avg Predicted H: 0.92
Episode 550 | Loss: 5.2637 | Avg Predicted H: 2.29
Episode 600 | Loss: 3.0742 | Avg Predicted H: 2.23
Episode 650 | Loss: 2.0179 | Avg Predicted H: 2.05
Episode 700 | Loss: 1.0514 | Avg Predicted H: 1.83
Episode 750 | Loss: 0.7371 | Avg Predicted H: 1.92
Episode 800 | Loss: 0.2734 | Avg Predicted H: 1.91
Episode 850 | Loss: 0.2289 | Avg Predicted H: 1.73
Episode 900 | Loss: 0.0851 | Av

In [40]:
env = TokenEnv(VOCABULARY, GOALS)
for goal in GOALS:
    print(f"\n==={goal}===")
    sp_goal = goal.split()
    start = ''
    for i in range(5):
        for token in VOCABULARY:
            h = model(env.state_to_tensor(f'{start} {token}')).item()
            print(f'{start} {token}: {h}')
        if i < len(sp_goal):
            start = f'{start} {sp_goal[i]}'
        else:
            continue




===I love you===
 I: 2.002877712249756
 love: 13.577937126159668
 you: 2.93107271194458
 too: 13.529239654541016
 are: 13.566859245300293
 really: 13.656574249267578
 incredible: 13.501972198486328
 I I: 12.578497886657715
 I love: 1.0062439441680908
 I you: 12.58565616607666
 I too: 12.633384704589844
 I are: 12.720728874206543
 I really: 12.68364143371582
 I incredible: 12.665746688842773
 I love I: 11.6272611618042
 I love love: 11.595969200134277
 I love you: 0.0
 I love too: 11.504511833190918
 I love are: 11.50888729095459
 I love really: 11.548382759094238
 I love incredible: 11.47976016998291
 I love you I: 1.5605533123016357
 I love you love: 2.4084537029266357
 I love you you: 2.1326770782470703
 I love you too: 2.9037396907806396
 I love you are: 2.640543222427368
 I love you really: 2.8950536251068115
 I love you incredible: 2.2461326122283936
 I love you I: 1.5605533123016357
 I love you love: 2.4084537029266357
 I love you you: 2.1326770782470703
 I love you too: 2.90373

In [41]:
# Save the model's weights to a file named 'token_heuristic.pt'
file_path = "token_heuristic.pt"
torch.save(model.state_dict(), file_path)
print(f"Model successfully saved to {file_path}")

Model successfully saved to token_heuristic.pt
